# 02 · PyFunc Custom Model — load_context + artifacts + 멀티 모델 래핑

## 왜 PyFunc 가 필요한가

Flavor logging (`mlflow.sklearn.log_model`) 은 **단일 native 객체** 직렬화에 최적화. 다음 케이스엔 PyFunc 필요:

- **여러 모델 / preprocessing 을 하나의 inference unit** 으로 묶을 때
- **flavor 가 없는 라이브러리** (ex. `surprise`, 자체 구현 모델)
- **post-processing** 이 모델 출력과 함께 패키지되어야 할 때
- **외부 artifact** (lookup table, embedding, config 등) 와 함께 배포할 때

## 핵심 컨트랙트: `PythonModel`

```python
class MyModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        # load_model() 시 한 번 실행 — 무거운 artifact 로드용
        ...
    def predict(self, context, model_input, params=None):
        # 매 요청마다 실행 — 가볍게 유지
        ...
```

In [ ]:
%run ./config

In [ ]:
import mlflow
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experiment_path)

## Step 1. 두 모델 학습 (RF + LR) — 앙상블 예시

In [ ]:
import pandas as pd, numpy as np, os, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

pdf = spark.table(f"{catalog}.{schema}.customers").toPandas()
FEATURES = ["age", "tenure_months", "monthly_charges", "total_charges", "support_tickets"]
X = pdf[FEATURES]; y = pdf["churned"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Random Forest (스케일 무관)
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42).fit(X_train, y_train)

# Logistic Regression (스케일 필요)
scaler = StandardScaler().fit(X_train)
lr = LogisticRegression(max_iter=500, random_state=42).fit(scaler.transform(X_train), y_train)

print("✓ RF + Scaler + LR 학습 완료")

## Step 2. Artifact 들을 Volume 에 저장

PyFunc 의 `artifacts` 파라미터는 **로컬 또는 UC Volume 경로**를 받아서 모델 디렉토리에 복사합니다.
서빙 시 MLflow 가 경로를 unpacked 모델 root 기준으로 재작성 → `context.artifacts["key"]` 로 접근.

In [ ]:
artifacts_dir = f"{volume_path}/02_pyfunc"
os.makedirs(artifacts_dir, exist_ok=True)

paths = {
    "rf":     f"{artifacts_dir}/rf.joblib",
    "scaler": f"{artifacts_dir}/scaler.joblib",
    "lr":     f"{artifacts_dir}/lr.joblib",
}
joblib.dump(rf,     paths["rf"])
joblib.dump(scaler, paths["scaler"])
joblib.dump(lr,     paths["lr"])

for k, p in paths.items():
    print(f"  {k:8s} → {p} ({os.path.getsize(p):,} bytes)")

## Step 3. PythonModel 정의 — 앙상블 + load_context + params

In [ ]:
import mlflow
from mlflow.pyfunc import PythonModel


class ChurnEnsemble(PythonModel):
    """RF + LR 앙상블 + threshold 후처리."""

    def load_context(self, context):
        # ↓ context.artifacts 는 dict[str, local_path] — MLflow 가 자동 resolve
        import joblib
        self.rf     = joblib.load(context.artifacts["rf"])
        self.scaler = joblib.load(context.artifacts["scaler"])
        self.lr     = joblib.load(context.artifacts["lr"])
        # ↓ context.model_config 는 log_model 시 model_config= 로 전달한 dict
        cfg = context.model_config or {}
        self.weight_rf  = cfg.get("weight_rf", 0.6)
        self.threshold  = cfg.get("threshold", 0.5)
        print(f"[load_context] weight_rf={self.weight_rf}, threshold={self.threshold}")

    def predict(self, context, model_input: pd.DataFrame, params=None):
        # params 는 호출 시점에 동적으로 받을 수 있는 값 (런타임 오버라이드)
        threshold = (params or {}).get("threshold", self.threshold)

        p_rf = self.rf.predict_proba(model_input)[:, 1]
        p_lr = self.lr.predict_proba(self.scaler.transform(model_input))[:, 1]
        p_ens = self.weight_rf * p_rf + (1 - self.weight_rf) * p_lr

        return pd.DataFrame({
            "probability": p_ens,
            "prediction":  (p_ens >= threshold).astype(int),
        })

## Step 4. 로컬 검증 — log 하기 전에 반드시 테스트

PyFunc 코드는 **로컬에서 먼저 검증**한 후 log 하세요. 서빙 컨테이너에서 디버깅하는 건 매우 비쌉니다.

In [ ]:
from types import SimpleNamespace

# load_context 가 받는 context 객체를 흉내 — artifacts dict + model_config dict
fake_context = SimpleNamespace(
    artifacts=paths,
    model_config={"weight_rf": 0.6, "threshold": 0.5},
)

model = ChurnEnsemble()
model.load_context(fake_context)
local_pred = model.predict(None, X_test.iloc[:5])
display(local_pred)

## Step 5. PyFunc log_model — `artifacts` + `model_config` + `params`

In [ ]:
from mlflow.models import infer_signature
from mlflow.types.schema import ParamSchema, ParamSpec, Schema
from mlflow.types import DataType

# signature with params — threshold 를 런타임에 받을 수 있음을 명시
sig = infer_signature(
    X_test.iloc[:5],
    local_pred,
    params={"threshold": 0.5},
)

with mlflow.start_run(run_name="pyfunc_ensemble"):
    info = mlflow.pyfunc.log_model(
        name="model",
        python_model=ChurnEnsemble(),
        artifacts=paths,            # ← Volume 경로의 joblib 파일들을 모델에 번들
        model_config={              # ← load_context 에서 context.model_config 로 접근
            "weight_rf": 0.6,
            "threshold": 0.5,
        },
        signature=sig,
        input_example=X_test.iloc[:5],
        registered_model_name=model_pyfunc,
        pip_requirements=[
            "scikit-learn==1.4.2",
            "joblib==1.4.0",
            "pandas==2.2.2",
            "numpy==1.26.4",
        ],
    )

print(f"✓ {model_pyfunc} v{info.registered_model_version}")

## Step 6. 등록된 모델로 load + 동적 threshold 사용

In [ ]:
from mlflow import MlflowClient
client = MlflowClient()
client.set_registered_model_alias(model_pyfunc, "Champion", info.registered_model_version)

loaded = mlflow.pyfunc.load_model(f"models:/{model_pyfunc}@Champion")

# 기본 threshold = 0.5 (model_config)
print("default (threshold=0.5):")
display(loaded.predict(X_test.iloc[:5]))

# 호출 시 동적으로 0.3 으로 override
print("\nruntime override (threshold=0.3):")
display(loaded.predict(X_test.iloc[:5], params={"threshold": 0.3}))

## 흔한 실수

| ❌ Anti-pattern | ✅ Right way |
| --- | --- |
| `predict` 안에서 joblib.load() | `load_context` 에서 한 번만 로드 |
| `self.path = "/Volumes/..."` 하드코딩 | `context.artifacts["key"]` 로 접근 |
| `log_model` 직후 검증 없이 register | `mlflow.pyfunc.load_model` 로 round-trip 테스트 |
| input_example 생략 | 무조건 포함 — signature + dep 추론 |

## 다음
→ **`03_dependencies`** — pip_requirements 옵션, private package, lock 전략